### Retrieval-Augmented Generation (RAG) Chatbot

This notebook provides a high-level, end-to-end overview of how to build a RAG chatbot. A RAG chatbot connects a large language model (LLM) to data sources like PDFs and websites, allowing the chatbot to give context-aware and accurate answers to frequently-asked questions.

### Prepare Data

- Scrape text from the UW-Madison DAPIR website.
- Clean text to remove whitespace and special characters.
- Chunk text into smaller, semantically-meaningful units (500 words).

In [1]:
# Import libraries
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse
import time
from typing import List, Dict
import re

# Specify constants (URL and max links to scrape)
# Start small for testing, setting MAX_LINKS to 10
# TODO: Expand scope by setting MAX_LINKS higher after testing
BASE_URL = "https://data.wisc.edu/"
MAX_LINKS = 10

In [2]:
def fetch_webpage(url: str) -> BeautifulSoup:
    """
    Fetch a webpage and return a BeautifulSoup object (parsed HTML).
    """
    # Use try-except for error handling
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        soup = BeautifulSoup(response.content, 'html.parser')
        return soup # Return parsed HTML
        
    except requests.exceptions.RequestException as e:
        print(f"Error fetching {url}: {e}") # Print error for debugging
        return None


def extract_links(soup: BeautifulSoup, base_url: str) -> List[str]:
    """
    Extract all links from the base url and convert them to absolute URLs.
    """
    # Initialize container for links
    links = []
    
    # Loop through all anchors (the <a> HTML element which defines hyperlinks)
    for anchor in soup.find_all('a', href=True):
        href = anchor['href']
        
        # Convert to absolute URL
        absolute_url = urljoin(base_url, href)

        # Include only links within the same domain
        if urlparse(absolute_url).netloc != urlparse(base_url).netloc:
            continue
        
        # Remove non-HTML content like PDFs and images
        # TODO: Are there ways to use non-HTML content for RAG context? Multi-modal RAG?
        if not any(absolute_url.endswith(ext) for ext in ['.pdf', '.jpg', '.png', '.zip']):
            if '#' not in absolute_url or absolute_url.split('#')[0] not in links:
                links.append(absolute_url.split('#')[0])
    
    # Remove duplicate links because a single page may link to the same URL multiple times
    seen = set()
    unique_links = []
    for link in links:
        if link not in seen:
            seen.add(link)
            unique_links.append(link)
    
    return unique_links

# Fetch the main webpage, extract all unique links, and subset links
main_soup = fetch_webpage(BASE_URL)
all_links = extract_links(main_soup, BASE_URL)
selected_links = all_links[:MAX_LINKS]

# Display selected links
print(f"Selected {len(selected_links)} links for RAG Chatbot context:")
for i, link in enumerate(selected_links, 1):
    print(f"  {i}. {link}")

Selected 10 links for RAG Chatbot context:
  1. https://data.wisc.edu/
  2. https://data.wisc.edu/about-us/
  3. https://data.wisc.edu/dapir-staff/
  4. https://data.wisc.edu/contact-us/
  5. https://data.wisc.edu/data-governance/
  6. https://data.wisc.edu/data-governance/data-domains-trustees-and-stewards/
  7. https://data.wisc.edu/data-governance/data-governance-council/
  8. https://data.wisc.edu/data-literacy/
  9. https://data.wisc.edu/data-governance/policies-procedures/
  10. https://data.wisc.edu/data-governance/data-classification/


In [3]:
def extract_text_content(soup: BeautifulSoup, url: str) -> Dict[str, str]:
    """
    Extract meaningful text content from a url.
    """
    # Remove script, style, and navigation elements
    for element in soup(['script', 'style', 'nav', 'footer', 'header', 'iframe']):
        element.decompose()
    
    # Extract title
    title = soup.find('title')
    title_text = title.get_text().strip() if title else "No Title"
    
    # Extract main content
    # Try to find main content area (common patterns)
    main_content = (
        soup.find('main') or 
        soup.find('article') or 
        soup.find('div', {'class': ['content', 'main-content', 'page-content']}) or
        soup.find('body')
    )
    
    # Extract text from headings (h1, h2, h3), paragraphs (p) and lists (li)
    text_elements = []
    
    if main_content:
        for tag in main_content.find_all(['h1', 'h2', 'h3', 'h4', 'p', 'li']):
            text = tag.get_text().strip()
            if text and len(text) > 10:  # Filter out very short snippets
                text_elements.append(text)
    
    combined_text = '\n\n'.join(text_elements)
    
    return {
        'url': url,
        'title': title_text,
        'content': combined_text,
        'word_count': len(combined_text.split())
    }


# Call functions to scrape content from selected links
scraped_data = []

# Fetch the page, extract content, and display results
for i, url in enumerate(selected_links, 1):
    soup = fetch_webpage(url)
    page_data = extract_text_content(soup, url)
    if page_data and page_data['word_count'] > 0:
        scraped_data.append(page_data)
        print(f"Scraped {page_data['word_count']} words from {url}")
    else:
        print(f"Error: No content extracted")
    time.sleep(1)

Scraped 370 words from https://data.wisc.edu/
Scraped 291 words from https://data.wisc.edu/about-us/
Scraped 245 words from https://data.wisc.edu/dapir-staff/
Scraped 87 words from https://data.wisc.edu/contact-us/
Scraped 490 words from https://data.wisc.edu/data-governance/
Scraped 407 words from https://data.wisc.edu/data-governance/data-domains-trustees-and-stewards/
Scraped 321 words from https://data.wisc.edu/data-governance/data-governance-council/
Scraped 363 words from https://data.wisc.edu/data-literacy/
Scraped 1693 words from https://data.wisc.edu/data-governance/policies-procedures/
Scraped 866 words from https://data.wisc.edu/data-governance/data-classification/


In [4]:
def clean_text(text: str) -> str:
    """
    Clean the scraped text content to prepare for chunking.
    """
    # Remove excessive whitespace
    text = re.sub(r'\s+', ' ', text)

    # Keep text, basic punctuation,and @ for email addresses
    text = re.sub(r'[^\w\s.,!?;:()\-\'\"@]+', '', text)
    
    # Remove repeated punctuation like '!!!' to '!'
    text = re.sub(r'([.,!?;:]){2,}', r'\1', text)
    
    # Return cleaned text
    return text.strip()

# See below about chunking at sentence boundaries
def chunk_text(text: str, chunk_size: int = 100, overlap: int = 10) -> List[Dict[str, any]]:
    """
    Split text into overlapping chunks (100 words each, overlapping by 10 words) to preserve words' context.
    """
    words = text.split()
    chunks = []
    
    for i in range(0, len(words), chunk_size - overlap):
        chunk_words = words[i:i + chunk_size]
        chunk_text = ' '.join(chunk_words)
        
        if len(chunk_words) > 10:
            chunks.append({
                'text': chunk_text,
                'word_count': len(chunk_words),
                'chunk_index': len(chunks)
            })
    
    return chunks

# Call functions to clean and chunk scraped data
processed_documents = []

for page_data in scraped_data:
    # Clean the content
    cleaned_content = clean_text(page_data['content'])
    
    # Chunk the content
    chunks = chunk_text(cleaned_content, chunk_size=500, overlap=100)
    
    # Add metadata to each chunk
    for chunk in chunks:
        processed_documents.append({
            'source_url': page_data['url'],
            'chunk_text': chunk['text'],
            'chunk_index': chunk['chunk_index'],
            'word_count': chunk['word_count']
        })
    
    # Display results
    print(f"Processed {page_data['url']} and created {len(chunks)} chunks")

Processed https://data.wisc.edu/ and created 1 chunks
Processed https://data.wisc.edu/about-us/ and created 1 chunks
Processed https://data.wisc.edu/dapir-staff/ and created 1 chunks
Processed https://data.wisc.edu/contact-us/ and created 1 chunks
Processed https://data.wisc.edu/data-governance/ and created 2 chunks
Processed https://data.wisc.edu/data-governance/data-domains-trustees-and-stewards/ and created 1 chunks
Processed https://data.wisc.edu/data-governance/data-governance-council/ and created 1 chunks
Processed https://data.wisc.edu/data-literacy/ and created 1 chunks
Processed https://data.wisc.edu/data-governance/policies-procedures/ and created 5 chunks
Processed https://data.wisc.edu/data-governance/data-classification/ and created 3 chunks


### Convert and Store Embeddings

- Convert text chunks into vector embeddings with a pre-trained model (HuggingFace sentence-transformers).
- Store embeddings and metadata in an open-source vector database (ChromaDB) to index chunks with embeddings and metadata like URLs.
- Whereas traditional databases use exact matching, vector databases use similarity search to find relevant content.

In [5]:
# Import libraries
from sentence_transformers import SentenceTransformer
import numpy as np

# Load pre-trained embeddings model from HuggingFace
# TODO: Experiment with different (free) models
model = SentenceTransformer('all-MiniLM-L6-v2')

# Extract text to generate embeddings for each chunk
chunk_texts = [doc['chunk_text'] for doc in processed_documents]

# Generate embeddings
embeddings = model.encode(
    chunk_texts,
    batch_size=32, # batch to process efficiently
    convert_to_numpy=True
)

# Add embeddings to processed documents
for i, doc in enumerate(processed_documents):
    doc['embedding'] = embeddings[i]

# Display results
print(f"Success! Embeddings added to documents.")
print(f"Example chunk text: {processed_documents[0]['chunk_text'][:100]}...")
print(f"Example embedding: [{processed_documents[0]['embedding'][:5]}...]")

/Users/tereuter/Library/Python/3.12/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Success! Embeddings added to documents.
Example chunk text: November's Badger Data Network: AI in institutional research In November, we welcome institutional r...
Example embedding: [[ 0.01462929 -0.08924618 -0.01165906  0.04414474  0.03398761]...]


In [6]:
# Import libraries
import chromadb
from chromadb.config import Settings

# Initialize ChromaDB client with persistent storage to save and reuse data
client = chromadb.PersistentClient(
    path="./chroma_db",
    settings=Settings(
        anonymized_telemetry=False,  # Disable telemetry to protect data privacy
        allow_reset=True # For development only! Do NOT allow in production because it can lead to accidental data loss!
    )
)

# Create or get a collection of related documents (like a table in SQL)
collection_name = "DAPIR_website_chunks"

# Delete existing collection if it exists (for fresh start during development)
try:
    client.delete_collection(name=collection_name)
except:
    pass

# Create new collection
collection = client.create_collection(
    name=collection_name,
    metadata={
        "description": "DAPIR website content chunks with embeddings",
        "embedding_model": "all-MiniLM-L6-v2",
        "created_date": time.strftime("%Y-%m-%d %H:%M:%S") # Store creation date
    }
)

# Prepare data for ChromaDB
# ChromaDB expects: ids, documents, embeddings, metadatas
ids = [f"chunk_{i}" for i in range(len(processed_documents))]
documents = [doc['chunk_text'] for doc in processed_documents]
embeddings_list = [doc['embedding'].tolist() for doc in processed_documents]

metadatas = [{
    'source_url': doc['source_url'],
    'chunk_index': doc['chunk_index'],
    'word_count': doc['word_count']
} for doc in processed_documents]

# Add documents to ChromaDB
collection.add(
    ids=ids,
    documents=documents,
    embeddings=embeddings_list,
    metadatas=metadatas
)

### Retrieval

- Convert user prompts into embeddings using the same, pre-trained model.
- Execute similarity search to find the most relevant text chunks.

In [7]:
def retrieve_relevant_chunks(query: str, collection, model, n_results: int = 5) -> Dict[str, any]:
    """
    Retrieve the most relevant text chunks from the vector database.
    """
    # Convert query to embedding using the same sentence-transformer model
    query_embedding = model.encode([query]).tolist()
    
    # Search the vector database for similar chunks
    results = collection.query(
        query_embeddings=query_embedding,
        n_results=n_results,
        include=['documents', 'metadatas', 'distances']
    )
    
    # Return query results
    return results

### Generation

- Pass the retrieved chunks as **context** to an open-source LLM.
- LLM prompts = user prompt + retrieved context + guardrails like "Answer only based on the provided context."
- Generate an answer grounded in the source documents.
- Include citations for source documents (URLs).

In [8]:
# Import libraries
from transformers import pipeline
import torch

# Check if GPU is available, otherwise use CPU
device = 0 if torch.cuda.is_available() else -1
device_name = "GPU" if device == 0 else "CPU"

# Initialize a text generation pipeline
generator = pipeline(
    "text2text-generation",
    # Set local model for RAG
    # google/flan-t5-base has ~250M parameters
    # google/flan-t5-large has ~780M parameters, so may perform better but more slowly
    model="google/flan-t5-large",
    device=device,
    max_length=512
)

Device set to use cpu


In [9]:
def truncate_to_sentence_boundary(text: str, tokenizer, max_tokens: int = 120) -> str:
    """
    Token-truncate text to max_tokens using the model's own tokenizer, then trim
    back to the last complete sentence so chunks don't end mid-sentence.
    """
    # Token-based truncation (reflects the model's actual input budget, unlike a raw character count)
    token_ids = tokenizer.encode(text, add_special_tokens=False)

    if len(token_ids) <= max_tokens:
        return text

    truncated_ids = token_ids[:max_tokens]
    truncated_text = tokenizer.decode(truncated_ids, skip_special_tokens=True)

    # Trim back to the last sentence boundary (., !, or ?) to avoid cutting mid-sentence
    last_boundary = max(
        truncated_text.rfind('.'),
        truncated_text.rfind('!'),
        truncated_text.rfind('?')
    )

    if last_boundary > 0:
        return truncated_text[:last_boundary + 1]

    # Fallback: no sentence boundary found within max_tokens, keep the token-truncated text
    return truncated_text + "..."


def format_context_with_sources(results: Dict[str, any], max_chunk_tokens: int = 120) -> tuple:
    """
    Format retrieved chunks into context text and extract unique sources (URLs).

    Each chunk is truncated individually (token-based, at a sentence boundary) BEFORE
    being concatenated, so every retrieved source is guaranteed some representation in
    the final context regardless of retrieval rank — an earlier, longer chunk will not
    block a later chunk that actually contains the answer.
    """
    context_parts = []
    sources = set()

    for i, (doc, metadata) in enumerate(zip(
        results['documents'][0],
        results['metadatas'][0]
    ), 1):
        # Truncate each source individually before concatenating
        truncated_doc = truncate_to_sentence_boundary(doc, generator.tokenizer, max_chunk_tokens)

        # Add context with source reference
        context_parts.append(f"[Source {i}]: {truncated_doc}")

        # Track unique sources
        url = metadata['source_url']
        sources.add(url)

    context_text = "\n\n".join(context_parts)
    sources_list = [{"url": url} for url in sources]

    return context_text, sources_list

In [13]:
def generate_answer(query: str, context: str, max_tokens: int = 100) -> str:
    """
    Generate a concise answer based solely on the provided context.
    """
    # Note: context is now truncated per-source (token-based, sentence-boundary-aware)
    # in format_context_with_sources, so no additional global truncation is needed here.
    try:
        # Generate the answer directly (no separate relevance-check pass)
        answer_prompt = f"""Context: {context}

Question: {query}

Instructions: Provide a direct, concise answer using ONLY information explicitly stated in the context above.

Answer:"""
        
        result = generator(
            answer_prompt,
            max_length=max_tokens,
            do_sample=False,
            num_beams=4,
            early_stopping=True
        )
        
        answer = result[0]['generated_text'].strip()
        
        # Fallback if model returns empty response
        if not answer or len(answer) < 3:
            return "I cannot answer this question based on the available information."
        
        return answer

    # Display error message    
    except Exception as e:
        error_msg = str(e)
        return f"Error generating response: {error_msg}"

In [14]:
# Define a function to call the above functions in sequence
def rag_chatbot(query: str, n_results: int = 5):
    """
    Execute the complete RAG pipeline: Retrieve context and generate answer.
    """
    # Display the user's question
    print(f"QUESTION: {query}")
    
    # Retrieve relevant chunks
    results = retrieve_relevant_chunks(
        query=query,
        collection=collection,
        model=model,
        n_results=n_results
    )
    
    # Format context and extract sources
    context, sources = format_context_with_sources(results)
    
    # Generate answer
    answer = generate_answer(
        query=query,
        context=context,
        max_tokens=500
    )
    
    # Display output on separate lines
    print(f"ANSWER: {answer}")
    sources_text = ", ".join([s['url'] for s in sources])
    print(f"SOURCE: {sources_text}\n")

### Model Evaluation

- Test the chatbot with sample FAQs.
- Are the answers accurate and helpful?

In [15]:
# Try different test questions here, including one that is intentionally out of scope
test_questions = [
    "What does DAPIR do?",
    "What should I do if I need help with Tableau?",
    "What is the current stock price of Google?"
]

# Loop over test questions and display the question and top answer for each one
for question in test_questions:
    rag_chatbot(question, n_results=3)

QUESTION: What does DAPIR do?
ANSWER: manages the structures needed to ensure the availability of accessible, usable, high-quality data
SOURCE: https://data.wisc.edu/data-literacy/, https://data.wisc.edu/contact-us/, https://data.wisc.edu/about-us/

QUESTION: What should I do if I need help with Tableau?
ANSWER: BIHelp-DAPIR@provost.wisc
SOURCE: https://data.wisc.edu/data-literacy/, https://data.wisc.edu/dapir-staff/, https://data.wisc.edu/contact-us/

QUESTION: What is the current stock price of Google?
ANSWER: unanswerable
SOURCE: https://data.wisc.edu/, https://data.wisc.edu/contact-us/, https://data.wisc.edu/data-governance/

